In [0]:
%sql
select * from dev.bronze.event_log_dp_bakehouse
order by timestamp desc

In [0]:
%sql
WITH latest_update AS (
  SELECT
    origin.pipeline_id,
    origin.update_id AS latest_update_id
  FROM dev.bronze.event_log_dp_bakehouse AS origin
  WHERE origin.event_type = 'create_update'
  ORDER BY timestamp DESC
  LIMIT 1
)
SELECT
  row_expectations.dataset as dataset,
  row_expectations.name as expectation,
  SUM(row_expectations.passed_records) as passing_records,
  SUM(row_expectations.failed_records) as failing_records
FROM
  (
    SELECT
      explode(
        from_json(
          details:flow_progress:data_quality:expectations,
          "array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>"
        )
      ) row_expectations
    FROM
      dev.bronze.event_log_dp_bakehouse,
      latest_update
    WHERE 1=1
      AND origin.update_id = latest_update.latest_update_id
  )
GROUP BY
  row_expectations.dataset,
  row_expectations.name;

In [0]:
%sql
with latest_update as (
  SELECT origin.update_id as id
    FROM dev.bronze.event_log_dp_bakehouse
    WHERE event_type = 'create_update'
    ORDER BY timestamp DESC
    limit 1 -- remove if you want all of the update_ids
)
SELECT
  details:flow_definition.output_dataset as flow_name,
  details:flow_definition.input_datasets as input_flow_names,
  details:flow_definition.flow_type as flow_type,
  details:flow_definition.schema, -- the schema of the flow
  details:flow_definition -- overall flow_definition object
FROM dev.bronze.event_log_dp_bakehouse inner join latest_update on origin.update_id = latest_update.id
WHERE details:flow_definition IS NOT NULL
ORDER BY timestamp;